# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data


A model that can estimate how much something costs, from its description.

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 2: Data Pre-processing

Today we'll rewrite the products into a standard format.  
LLMs are great at this!

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business value of Data Pre-processing / Re-writing</h2>
            <span style="color:#181;">LLMs have made it simple to do something that was considered impossible only a few years ago.
            This approach can be applied to almost any business vertical, and it's similar to the advanced techniques
            we used on Week 5.</span>
        </td>
    </tr>
</table>

In [2]:
from litellm import completion
from dotenv import load_dotenv
from pricer.items import Item
from tqdm import tqdm
import os
import json

load_dotenv(override=True)

True

# The next cell is where you choose Dataset

Use `LITE_MODE = True` for the free, fast version with training data size of 20,000

USe `LITE_MODE =  False` for the powerful, full version with training data size of 800,000

## For this lab

You can skip altogether and load the dataset from HuggingFace: $0

You can run pre-processing for the lite dataset: under $1

You can run pre-processing for the full dataset: $30

In [3]:
LITE_MODE = True

MODEL = "ollama/llama3.2:1b"
API_BASE = "http://localhost:11434"

username = "andrevsilva"
dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"

In [4]:
train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f"Loaded {len(items):,} items")
print(items[0])

Loaded 22,000 items
title='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)' category='Tools_and_Home_Improvement' price=64.3 full='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\n[\'From the Manufacturer\', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4" minimum center to center door prep required for this two piece model.\', \'Lifetime Mechanical and Finish Warranty\']\n{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Wei

In [5]:
print(items[0])

title='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)' category='Tools_and_Home_Improvement' price=64.3 full='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\n[\'From the Manufacturer\', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4" minimum center to center door prep required for this two piece model.\', \'Lifetime Mechanical and Finish Warranty\']\n{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Weight": "1.5 pounds", 

In [6]:
for index, item in enumerate(items):
    item.id = index

In [7]:
print(items[1000])

title='Cardone 12-10230 Remanufactured Anti-Lock Brake ABS Control Unit Module, EBCM (Renewed)' category='Automotive' price=222.09 full='Cardone 12-10230 Remanufactured Anti-Lock Brake ABS Control Unit Module, EBCM (Renewed)\n[\'CARDONE Remanufactured ABS Control Modules are designed to meet or exceed O.E. performance. Reverse engineering provides insight into how and why the unit originally failed, allowing our engineers to identify and correct original design flaws. Every CARDONE unit goes through stringent testing, ensuring like-new performance and quick reaction time when traction control is required. Air-decay testing ensures unit is void of brake fluid; high-pressure hydraulic testing ensures zero leakage.\']\n[\'As a remanufactured Original Equipment part, this unit assures a perfect vehicle fit\', \'Worn-out, missing or non-functioning components are replaced with new or rebuilt components, where necessary\', \'Critical components are re-soldered, where necessary, to ensure sup

In [8]:
SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

In [23]:
print(items[3].full)

Hollyland Mars 300 Pro Enhanced Wireless Video Transmitter & Receiver 300Ft Range 1080p HDMI 5G 0.08S Low Latency APP Support iOS Android [2 Battery Pack & AC Charger Bundle]
['KEY ', 'Built-In Antennas - Built-In Antennas for Easy Setups, Efficient Shooting, and Different Shooting Applications also Includes External Antennas300ft Transmission Range with 0.08s Minimum Latency - Up to 300ft Hassle-Free and Reliable Range for Wireless Video and Audio Transmission. 0.08S Lowest Achievable Latency for Real-Time MonitoringThumbwheel Switch Menu - Better Using Experience with the Multi-Functional Clickable Thumbwheel SwitchMultiple Power Options - Supports 5-12V Wide Voltage Power Supply, Including Various L-Series Batteries, Different Power Banks, and Type-C (5V/2A) ChargingHDMI In & Loopout - HDMI Input and loopout on TX, and Dual HDMI Outputs on RXSide OLED - Side OLED Provides Easy Access to Power Status, Channel Scan, and Other OLED Display Information for Both Vertical or Horizontal In

In [24]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="ollama/llama3.2:1b", api_base="http://localhost:11434")
print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")

**Simplistic Security Hardware**
The Schlage F59 AND 613 Andover Interior Knob with Deadbolt is a precision-engineered security handle set that provides reliable deadbolting and assurance of home security. Its oil rubbed bronze finish offers a durable and attractive appearance.

Input tokens: 406
Output tokens: 56
Cost: 0.000 cents


In [25]:
MODEL = "ollama/llama3.2:1b"

In [26]:
def make_jsonl(item):
    body = {"model": MODEL, "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": item.full}], "reasoning_effort": "low"}
    line = {"custom_id": item.id, "method": "POST", "url": "/v1/chat/completions", "body": body}
    return json.dumps(line)

In [27]:
def make_file(start, end, filename):
    batch_file = filename
    with open(batch_file, "w") as f:
        for i in range(start, end):
            f.write(make_jsonl(items[i]))
            f.write("\n")

In [28]:
make_jsonl(items[0])

'{"custom_id": 0, "method": "POST", "url": "/v1/chat/completions", "body": {"model": "ollama/llama3.2:1b", "messages": [{"role": "system", "content": "Create a concise description of a product. Respond only in this format. Do not include part numbers.\\nTitle: Rewritten short precise title\\nCategory: eg Electronics\\nBrand: Brand name\\nDescription: 1 sentence description\\nDetails: 1 sentence on features"}, {"role": "user", "content": "Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\\n[\'From the Manufacturer\', \\"When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid\\"]\\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4\\" minimum center to center door prep required for this two piece mod

In [29]:
make_file(0, 1000, "jsonl/0_1000.jsonl")

In [30]:
import json
import requests
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
import os

OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL = "llama3.2:1b"

def process_line(line):
    data = json.loads(line)

    payload = {
        "model": MODEL,
        "messages": data["body"]["messages"],
        "options": {
            "temperature": 0.3
        },
        "stream": False
    }

    response = requests.post(OLLAMA_URL, json=payload)
    return response.status_code  # you can return response.json() if needed


if __name__ == "__main__":
    # Read all lines first
    with open("jsonl/0_1000.jsonl", "r") as f:
        lines = f.readlines()

    num_workers = os.cpu_count()  # use all cores

    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        futures = [executor.submit(process_line, line) for line in lines]

        for _ in tqdm(as_completed(futures), total=len(futures), desc="Processing", unit="req"):
            pass

    print("Batch processing completed.")

Processing: 100%|██████████| 1000/1000 [1:28:14<00:00,  5.29s/req] 

Batch processing completed.


In [31]:
OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL = "llama3.2:1b"


def process_line(line):
    data = json.loads(line)

    payload = {
        "model": MODEL,
        "messages": data["body"]["messages"],
        "options": {
            "temperature": 0.3
        },
        "stream": False
    }

    response = requests.post(OLLAMA_URL, json=payload)
    result = response.json()

    return {
        "custom_id": data["custom_id"],
        "response": result["message"]["content"]
    }


def process_batch():
    with open("jsonl/0_1000.jsonl", "r") as infile:
        lines = infile.readlines()

    num_workers = os.cpu_count()

    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        futures = [executor.submit(process_line, line) for line in lines]

        with open("jsonl/batch_results.jsonl", "w") as outfile:
            for future in as_completed(futures):
                output_record = future.result()
                outfile.write(json.dumps(output_record) + "\n")


if __name__ == "__main__":
    process_batch()

In [10]:
with open("jsonl/batch_results.jsonl", "r") as f:
    for line in f:
        json_line = json.loads(line)
        id = json_line["custom_id"]
        summary = json_line["response"]
        items[id].summary = summary

In [12]:
print(items[999].summary)

Here is a concise description of the product:

The Rewritten short precise title: Supplying Demand DG94-00520A Gas Range Hot Surface Igniter Assembly Replacement

This replacement part is designed to replace the hot surface igniter assembly on gas ranges. It includes an igniter, block, cage, leads with female pins, and lights gas flames but will not ignite if the ignitor has failed due to improper heating.


In [13]:
print(items[10].summary)

MGP Caliper Covers - Front and Rear Brake Caliper Covers for Jeep Grand Cherokee (42020s) 
Made with 6061T6 Aerospace Grade Aluminum, Powder Coated for Durability
Quick and Easy Installation; about 1 hour


In [ ]:
for index, item in enumerate(items):
    if not item.summary:
        print(index)

In [15]:
# Remove the fields that we don't need in the hub

for item in items:
    item.full = None
    item.id = None

In [16]:
full = f"{username}/items_full"
lite = f"{username}/items_lite"

if LITE_MODE:
    train = items[:799]
    val = items[800:899]
    test = items[900:999]
    Item.push_to_hub(lite, train, val, test)


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/729 [00:00<?, ?B/s]